## Data preprocessing 

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
import joblib
from sklearn.linear_model import LogisticRegression


In [3]:
# read, load the data 
df = pd.read_csv("../data/processed/preprocessing_input.csv")

In [4]:
# Features
X = df.drop(columns=["Mental_Health_Risk"])

# Target 
y = df['Mental_Health_Risk']


In [5]:
# Split data (Train 80%, Test 20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f"Train Data : {X_train.shape[0]} rows, {X_train.shape[1]} columns")
print(f"Test Data : {X_test.shape[0]} rows, {X_test.shape[1]} columns")

Train Data : 1600 rows, 15 columns
Test Data : 400 rows, 15 columns


In [6]:
# Check missing values in training data
missing_train = X_train.isnull().sum()

missing_train = missing_train[missing_train > 0]

print("Missing Values in X_train:")
print(missing_train)

Missing Values in X_train:
Sleep_Hours           54
GPA                   36
Social_Media_Hours    48
Financial_Stress      76
Family_Support        36
Attendance_Rate       39
Diet_Quality          55
dtype: int64


In [7]:
# Check missing values in testing data
missing_test = X_test.isnull().sum()

missing_test = missing_test[missing_test > 0]

print("Missing Values in X_test:")
print(missing_test)

Missing Values in X_test:
Sleep_Hours            8
GPA                    2
Social_Media_Hours    10
Financial_Stress      15
Family_Support        18
Attendance_Rate        8
Diet_Quality          14
dtype: int64


In [9]:
print("Train Missing Values:", X_train.isnull().sum().sum())
print("Test Missing Values:", X_test.isnull().sum().sum())

Train Missing Values: 344
Test Missing Values: 75


##### Detect Outliers

In [10]:
num_cols = X_train.select_dtypes(include="number").columns

Q1 = X_train[num_cols].quantile(0.25)
Q3 = X_train[num_cols].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = ((X_train[num_cols] < lower_bound) | (X_train[num_cols] > upper_bound)).sum()

print(outliers.sort_values(ascending=False))

Attendance_Rate         10
Social_Media_Hours      10
Sleep_Hours              9
Study_Hours_Per_Week     7
Screen_Time_Hours        7
GPA                      5
Age                      2
Year_of_Study            0
Financial_Stress         0
Family_Support           0
dtype: int64


In [11]:
for col in num_cols:

    mask = (
        (X_train[col] < lower_bound[col]) |
        (X_train[col] > upper_bound[col])
    )

    if mask.sum() > 0:
        print(f"\n--- {col} ---")
        print(sorted(X_train.loc[mask, col].tolist()))


--- Age ---
[28, 28]

--- Sleep_Hours ---
[3.6, 3.7, 9.8, 9.9, 9.9, 10.0, 10.0, 10.0, 10.0]

--- Study_Hours_Per_Week ---
[2.4, 2.7, 3.2, 3.6, 3.9, 37.5, 38.3]

--- GPA ---
[1.5, 3.53, 3.57, 3.64, 3.66]

--- Social_Media_Hours ---
[2.1, 2.9, 3.0, 9.1, 9.2, 9.3, 9.4, 9.5, 10.0, 10.0]

--- Screen_Time_Hours ---
[3.3, 3.4, 3.8, 3.8, 3.8, 13.6, 14.0]

--- Attendance_Rate ---
[54.4, 61.1, 61.8, 61.9, 62.4, 62.5, 62.5, 62.8, 63.2, 65.3]


In [12]:
# check the variables that are categorical in the training data
cat_cols = X_train.select_dtypes(
    include=['object', 'str', 'category']
).columns.tolist()

print(cat_cols)

['Gender', 'Physical_Activity', 'Part_Time_Job', 'Diet_Quality', 'Previous_Mental_Health_History']


In [ ]:

# group ordinal cols 
ordinal_cols = [
    'Physical_Activity',
    'Diet_Quality'
]

ordinal_mapping = [
    ['Low', 'Moderate', 'High'],
    ['Poor', 'Average', 'Good']
]


nominal_cols = [ 
    'Gender',
    "Part_Time_Job",
    "Previous_Mental_Health_History",
]



#### — Check Class Distribution

In [16]:
print("Training Target Distribution:")
print(y_train.value_counts())

print("\nTraining Target Percentage:")
print(y_train.value_counts(normalize=True) * 100)

Training Target Distribution:
Mental_Health_Risk
High      544
Medium    528
Low       528
Name: count, dtype: int64

Training Target Percentage:
Mental_Health_Risk
High      34.0
Medium    33.0
Low       33.0
Name: proportion, dtype: float64


##### **After Checking classes the classes are Already balanced**
    => No SMOTE Needed

In [19]:
numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

ordinal_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OrdinalEncoder(categories=ordinal_mapping)),
    ("scaler", StandardScaler())
])

nominal_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(sparse_output=False, handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ("numerical", numerical_pipeline, num_cols),
    ("ordinal", ordinal_pipeline, ordinal_cols),
    ("nominal", nominal_pipeline, nominal_cols)
])

In [ ]:
# test the preprocessor on the training and test data
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# print(X_train_processed.mean().round(2))
# print(X_train_processed.std().round(2))


array([[-0.0637065 , -0.47163633, -1.40418192, ...,  0.        ,
         1.        ,  0.        ],
       [ 0.87574418, -1.2323401 , -1.6760358 , ...,  0.        ,
         1.        ,  0.        ],
       [-1.00315718, -0.47163633,  0.40817727, ...,  0.        ,
         1.        ,  0.        ],
       ...,
       [-1.94260785,  1.81047496,  0.04570543, ...,  1.        ,
         1.        ,  0.        ],
       [ 0.40601884,  1.04977119, -0.7698562 , ...,  0.        ,
         1.        ,  0.        ],
       [-0.53343184, -1.2323401 , -0.95109212, ...,  0.        ,
         0.        ,  1.        ]], shape=(1600, 19))

In [21]:
artifacts = {
    "preprocessor" : preprocessor,
    "X_train" : X_train,
    "X_test" : X_test,
    "y_train" : y_train,
    "y_test" : y_test
}
joblib.dump(artifacts, "../models/preprocessor_artifacts.joblib")
print("Preprocessing artifacts saved successfully!")

Preprocessing artifacts saved successfully!
